# Modelado Predictivo — Dengue SIN Signos de Alarma (DSSA)
## DSSA · 15,734 casos · 3.9% hospitalizados

---

El desbalance es severo: solo el 3.9% de los pacientes se hospitalizan.
Esto cambia fundamentalmente la estrategia de modelado respecto al DCSA:

| Aspecto | DCSA (con signos) | DSSA (sin signos) |
|---------|-------------------|-------------------|
| Hospitalizacion | 64.8% | 3.9% |
| Desbalance | Moderado (65/35) | Severo (96/4) |
| Metrica primaria | F1-score | AUC-PR |
| SMOTE k_neighbors | 5 | 3 |

**Por que AUC-PR y no F1?** Con desbalance 96/4, un modelo que predice
siempre 'no hospitalizado' tiene accuracy=96% y AUC-ROC>0.5 pero
AUC-PR≈0.04 (igual a la prevalencia). AUC-PR es implacable con los
modelos que ignoran la clase minoritaria.

**Siguiente:** `04_clustering/04_analisis_no_supervisado.ipynb`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import joblib
import importlib
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, balanced_accuracy_score,
    precision_score, recall_score, average_precision_score,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    classification_report, make_scorer)
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
if importlib.util.find_spec('catboost') is None:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'catboost'])
from catboost import CatBoostClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'legend.frameon': False,
})

COLORS = {
    'primary': '#2C7BB6', 'secondary': '#D7191C',
    'accent': '#FDAE61', 'neutral': '#7F7F7F',
    'con_signos': '#E74C3C', 'sin_signos': '#3498DB',
}
SIGNOS_ALARMA      = ['DOLOR_ABDO','VOMITO','DIARREA','SOMNOLENCI','HIPOTENSIO',
                      'HEPATOMEG','HEM_MUCOSA','HIPOTERMIA','AUM_HEMATO','CAIDA_PLAQ','ACUM_LIQUI']
SINTOMAS_GENERALES = ['FIEBRE','CEFALEA','DOLRRETROO','MALGIAS','ARTRALGIA','ERUPCIONR']
TARGET             = 'TARGET_HOSPITALIZACION_ORIGINAL'
RANDOM_STATE       = 42

ROOT        = Path(r'D:\Documentos\GitHub\dengue')  # <-- edita aqui si es necesario
OUT_FIG     = ROOT / 'outputs' / 'figures'
OUT_TAB     = ROOT / 'outputs' / 'tables'
INTERIM     = ROOT / 'data' / 'interim'
CHECKPOINTS = ROOT / 'outputs' / 'checkpoints'
for p in [OUT_FIG, OUT_TAB, INTERIM, CHECKPOINTS]:
    p.mkdir(parents=True, exist_ok=True)

df = joblib.load(INTERIM / 'datos_sin_signos.pkl')
print(f'DSSA — Dengue sin signos de alarma')
print(f'  Casos      : {len(df):,}')
print(f'  Hosp.      : {df[TARGET].mean():.1%}  <- desbalance severo')
print(f'  No hosp.   : {1-df[TARGET].mean():.1%}')
print()
print('ATENCION: desbalance 96/4 — AUC-PR es la metrica primaria, no F1')

In [ ]:
import gc

def checkpoint_existe(path):
    return Path(path).exists()

def guardar_checkpoint(obj, path, nombre=''):
    joblib.dump(obj, path)
    mb = Path(path).stat().st_size / 1e6
    print(f'  Guardado {nombre}: {Path(path).name} ({mb:.1f} MB)')

def limpiar_memoria(*objetos):
    import gc
    for obj in objetos:
        try: del obj
        except: pass
    gc.collect()

def memoria_usada():
    import psutil, os
    mb = psutil.Process(os.getpid()).memory_info().rss / 1e6
    print(f'  RAM en uso: {mb:.0f} MB')

try:
    import psutil
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'psutil'])
    import psutil

print('Utilidades listas')

## 1. Preparación de features y partición

In [ ]:
vars_clinicas = SINTOMAS_GENERALES + SIGNOS_ALARMA
vars_demo     = [v for v in ['SEXO_','DEF_CLAS_EDAD','TIP_SS_','ESTRATO_','GP_GESTAN','GRUPO_ETARIO']
                 if v in df.columns]
vars_tiempo   = [v for v in ['AÑO','SEMANA'] if v in df.columns]
FEATURES      = vars_clinicas + vars_demo + vars_tiempo
CP_PARTICION  = CHECKPOINTS / 'particion_DSSA.pkl'

if checkpoint_existe(CP_PARTICION):
    print('Cargando particion desde checkpoint...')
    X_train, X_test, y_train, y_test, FEATURES = joblib.load(CP_PARTICION)
    vars_clinicas = SINTOMAS_GENERALES + SIGNOS_ALARMA
    vars_demo     = [v for v in FEATURES if v not in vars_clinicas + ['AÑO','SEMANA']]
    vars_tiempo   = [v for v in FEATURES if v in ['AÑO','SEMANA']]
else:
    df_m = df[FEATURES + [TARGET]].copy()
    for v in vars_clinicas:
        df_m[v] = df_m[v].map({1: 1, 2: 0})
    for v in vars_demo:
        if str(df_m[v].dtype) in ('object',) or str(df_m[v].dtype).startswith('string'):
            df_m[v] = pd.Categorical(df_m[v]).codes.astype(float)
            df_m[v] = df_m[v].replace(-1, np.nan)
    for col in FEATURES:
        df_m[col] = pd.to_numeric(df_m[col], errors='coerce').astype(np.float64)
    df_m = df_m.dropna(subset=[TARGET])
    X = df_m[FEATURES]; y = df_m[TARGET].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)
    guardar_checkpoint((X_train, X_test, y_train, y_test, FEATURES),
        CP_PARTICION, 'particion DSSA')
    limpiar_memoria(df_m, X, y)

print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')
print(f'Prev. train: {y_train.mean():.1%} | test: {y_test.mean():.1%}')
memoria_usada()

In [ ]:
preprocesador = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler())
    ]), vars_demo + vars_tiempo),
    ('bin', SimpleImputer(strategy='most_frequent'), vars_clinicas),
], remainder='drop')

# SMOTE con k_neighbors=3 para desbalance severo (pocos positivos)
# k=5 puede generar ejemplos fuera del soporte cuando la clase positiva es muy pequeña
def make_pipe(clf, smote=True):
    pasos = [('prep', preprocesador)]
    if smote:
        pasos.append(('smote', SMOTE(random_state=RANDOM_STATE, k_neighbors=3)))
    pasos.append(('clf', clf))
    return ImbPipeline(pasos)

print('Preprocesador listo')

## 2. Baselines

Con desbalance 96/4 el Dummy que predice 'no hospitalizado' tiene
accuracy=96% pero AUC-PR=0.04. Ilustra por qué accuracy es inútil
en datos muy desbalanceados.

In [ ]:
CP_BASELINE = CHECKPOINTS / 'baseline_DSSA.pkl'
if checkpoint_existe(CP_BASELINE):
    print('Baselines ya entrenados — cargando...')
    dummy, dt = joblib.load(CP_BASELINE)
else:
    dummy = make_pipe(DummyClassifier(strategy='most_frequent'), smote=False)
    dt    = make_pipe(DecisionTreeClassifier(max_depth=3, min_samples_leaf=30,
        class_weight='balanced', random_state=RANDOM_STATE), smote=False)
    dummy.fit(X_train, y_train)
    dt.fit(X_train, y_train)
    guardar_checkpoint((dummy, dt), CP_BASELINE, 'baselines')

for nombre, pipe in [('Dummy', dummy), ('Decision Tree depth=3', dt)]:
    y_pred = pipe.predict(X_test)
    print(f'\n=== {nombre} ===')
    print(classification_report(y_test, y_pred,
        target_names=['No hosp.', 'Hospitalizado']))
    if hasattr(pipe, 'predict_proba'):
        y_pr = pipe.predict_proba(X_test)[:, 1]
        print(f'AUC-PR: {average_precision_score(y_test, y_pr):.4f}')
limpiar_memoria()
memoria_usada()

## 3. Portafolio completo — validación cruzada 5-fold

Métrica primaria: **AUC-PR** (Average Precision Score).
Un modelo que no supera la prevalencia (3.9%) en AUC-PR
no aporta valor clínico.

In [ ]:
CP_CV = CHECKPOINTS / 'cv_DSSA.pkl'

portafolio_cv = {
    'Dummy':             make_pipe(DummyClassifier(strategy='most_frequent'), smote=False),
    'Decision Tree':     make_pipe(DecisionTreeClassifier(max_depth=3,
                             class_weight='balanced', random_state=RANDOM_STATE), smote=False),
    'Logistic Reg.':     make_pipe(LogisticRegression(max_iter=500, C=1.0,
                             random_state=RANDOM_STATE, n_jobs=1)),
    'Random Forest':     make_pipe(RandomForestClassifier(n_estimators=100,
                             max_depth=8, random_state=RANDOM_STATE, n_jobs=1)),
    'Extra Trees':       make_pipe(ExtraTreesClassifier(n_estimators=100,
                             max_depth=8, random_state=RANDOM_STATE, n_jobs=1)),
    'Gradient Boosting': make_pipe(GradientBoostingClassifier(n_estimators=100,
                             learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE)),
    'XGBoost':           make_pipe(XGBClassifier(n_estimators=100, learning_rate=0.05,
                             max_depth=5, eval_metric='logloss', tree_method='hist',
                             random_state=RANDOM_STATE, n_jobs=1, verbosity=0)),
    'LightGBM':          make_pipe(LGBMClassifier(n_estimators=100, learning_rate=0.05,
                             max_depth=6, num_leaves=31,
                             random_state=RANDOM_STATE, n_jobs=1, verbose=-1)),
    'CatBoost':          make_pipe(CatBoostClassifier(iterations=100, learning_rate=0.05,
                             depth=6, random_seed=RANDOM_STATE,
                             thread_count=1, verbose=0)),
}

portafolio_final = {
    'Dummy':             make_pipe(DummyClassifier(strategy='most_frequent'), smote=False),
    'Decision Tree':     make_pipe(DecisionTreeClassifier(max_depth=3,
                             class_weight='balanced', random_state=RANDOM_STATE), smote=False),
    'Logistic Reg.':     make_pipe(LogisticRegression(max_iter=1000, C=1.0,
                             random_state=RANDOM_STATE)),
    'Random Forest':     make_pipe(RandomForestClassifier(n_estimators=300,
                             max_depth=10, random_state=RANDOM_STATE, n_jobs=1)),
    'Extra Trees':       make_pipe(ExtraTreesClassifier(n_estimators=300,
                             max_depth=10, random_state=RANDOM_STATE, n_jobs=1)),
    'Gradient Boosting': make_pipe(GradientBoostingClassifier(n_estimators=200,
                             learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE)),
    'XGBoost':           make_pipe(XGBClassifier(n_estimators=300, learning_rate=0.05,
                             max_depth=5, eval_metric='logloss', tree_method='hist',
                             random_state=RANDOM_STATE, n_jobs=1, verbosity=0)),
    'LightGBM':          make_pipe(LGBMClassifier(n_estimators=300, learning_rate=0.05,
                             max_depth=6, num_leaves=31,
                             random_state=RANDOM_STATE, n_jobs=1, verbose=-1)),
    'CatBoost':          make_pipe(CatBoostClassifier(iterations=300, learning_rate=0.05,
                             depth=6, random_seed=RANDOM_STATE,
                             thread_count=1, verbose=0)),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    'average_precision': make_scorer(average_precision_score, needs_proba=True),
    'roc_auc':           make_scorer(roc_auc_score, needs_proba=True),
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'f1':                make_scorer(f1_score, pos_label=1),
    'recall':            make_scorer(recall_score, pos_label=1),
    'precision':         make_scorer(precision_score, pos_label=1, zero_division=0),
}

if checkpoint_existe(CP_CV):
    print('CV ya calculada — cargando...')
    resultados_cv = joblib.load(CP_CV)
    for nombre, res in resultados_cv.items():
        print(f"  {nombre}: AUC-PR={res['average_precision']['mean']:.3f}")
else:
    resultados_cv = {}
    for nombre, pipe in portafolio_cv.items():
        CP_MOD = CHECKPOINTS / f'cv_DSSA_{nombre.replace(" ","_").replace(".","")}.pkl'
        if checkpoint_existe(CP_MOD):
            print(f'  {nombre}: ya calculado, cargando...')
            resultados_cv[nombre] = joblib.load(CP_MOD)
        else:
            print(f'  Entrenando {nombre}...', end=' ', flush=True)
            cv_res = cross_validate(pipe, X_train, y_train,
                cv=cv, scoring=scoring, n_jobs=1)
            resultados_cv[nombre] = {
                m: {'mean': cv_res[f'test_{m}'].mean(),
                    'std':  cv_res[f'test_{m}'].std()}
                for m in scoring
            }
            guardar_checkpoint(resultados_cv[nombre], CP_MOD, nombre)
            print(f"AUC-PR={resultados_cv[nombre]['average_precision']['mean']:.3f}")
        limpiar_memoria()
        memoria_usada()
    guardar_checkpoint(resultados_cv, CP_CV, 'CV completa DSSA')

memoria_usada()

## 4. Tabla y gráfico comparativo

In [ ]:
filas = []
for nombre, res in resultados_cv.items():
    filas.append({
        'Modelo':    nombre,
        'AUC_PR':    round(res['average_precision']['mean'], 4),
        'AUC_PR_std':round(res['average_precision']['std'], 4),
        'AUC_ROC':   round(res['roc_auc']['mean'], 4),
        'Bal_Acc':   round(res['balanced_accuracy']['mean'], 4),
        'F1':        round(res['f1']['mean'], 4),
        'Recall':    round(res['recall']['mean'], 4),
        'Precision': round(res['precision']['mean'], 4),
    })
# Ordenar por AUC-PR (metrica primaria DSSA)
tabla_cv = pd.DataFrame(filas).sort_values('AUC_PR', ascending=False)
display(tabla_cv)
tabla_cv.to_csv(OUT_TAB / 'cv_DSSA.csv', index=False)

colores_m = ['#95A5A6','#BDC3C7','#2980B9','#1A9850','#66C2A5',
             '#F46D43','#D73027','#A50026','#762A83']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
x  = np.arange(len(tabla_cv))
w  = 0.35
ax.bar(x - w/2, tabla_cv['AUC_PR'], w, yerr=tabla_cv['AUC_PR_std'],
    color=colores_m[:len(tabla_cv)], alpha=0.88, capsize=4,
    label='AUC-PR (primaria)', edgecolor='white')
ax.bar(x + w/2, tabla_cv['AUC_ROC'], w,
    color=colores_m[:len(tabla_cv)], alpha=0.45, capsize=4,
    label='AUC-ROC', edgecolor='white', hatch='//')
# Prevalencia como piso de AUC-PR
prev = y_train.mean()
ax.axhline(prev, color=COLORS['neutral'], linestyle=':',
    linewidth=1.5, label=f'Prevalencia (piso AUC-PR): {prev:.1%}')
ax.set_xticks(x)
ax.set_xticklabels(tabla_cv['Modelo'], rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Puntuacion (CV 5-fold)')
ax.set_title('A. AUC-PR y AUC-ROC | DSSA')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

ax = axes[1]
ax.scatter(tabla_cv['Recall'], tabla_cv['Precision'],
    s=140, c=colores_m[:len(tabla_cv)], zorder=5,
    edgecolors='white', linewidth=0.8)
for i, (_, row) in enumerate(tabla_cv.iterrows()):
    ax.annotate(row['Modelo'], (row['Recall'], row['Precision']),
        fontsize=7.5, xytext=(4, 4), textcoords='offset points')
ax.axhline(prev, color=COLORS['neutral'], linestyle='--', alpha=0.5,
    label=f'Prevalencia: {prev:.1%}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('B. Precision vs Recall | DSSA')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('Comparacion de modelos — CV 5-fold estratificada\nDSSA: dengue sin signos de alarma',
    fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIG / 'sin_signos_cv_comparacion.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 5. Evaluación final en test set

In [ ]:
CP_EVAL = CHECKPOINTS / 'eval_test_DSSA.pkl'

filas = []
for nombre, res in resultados_cv.items():
    filas.append({'Modelo': nombre,
        'AUC_PR':    round(res['average_precision']['mean'], 4),
        'AUC_PR_std':round(res['average_precision']['std'], 4),
        'AUC_ROC':   round(res['roc_auc']['mean'], 4),
        'Bal_Acc':   round(res['balanced_accuracy']['mean'], 4),
        'F1':        round(res['f1']['mean'], 4),
        'Recall':    round(res['recall']['mean'], 4),
        'Precision': round(res['precision']['mean'], 4),
    })
tabla_cv = pd.DataFrame(filas).sort_values('AUC_PR', ascending=False)
display(tabla_cv)
tabla_cv.to_csv(OUT_TAB / 'cv_DSSA.csv', index=False)

if checkpoint_existe(CP_EVAL):
    print('Evaluacion final ya calculada — cargando...')
    saved = joblib.load(CP_EVAL)
    tabla_test = saved['tabla_test']
    entrenados = saved['entrenados']
    mejor      = saved['mejor']
    display(tabla_test)
else:
    top3 = (tabla_cv[~tabla_cv['Modelo'].isin(['Dummy','Decision Tree'])]
        .head(3)['Modelo'].tolist())
    print(f'Top 3 por AUC-PR: {top3}')
    entrenados, res_test = {}, []
    for nombre in top3:
        pipe = portafolio_final[nombre]
        print(f'  Entrenando {nombre} (parametros completos)...')
        pipe.fit(X_train, y_train)
        y_prob = pipe.predict_proba(X_test)[:, 1]
        y_pred = pipe.predict(X_test)
        res_test.append({
            'Modelo':    nombre,
            'AUC_PR':    round(average_precision_score(y_test, y_prob), 4),
            'AUC_ROC':   round(roc_auc_score(y_test, y_prob), 4),
            'F1':        round(f1_score(y_test, y_pred), 4),
            'Recall':    round(recall_score(y_test, y_pred), 4),
            'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
            'Bal_Acc':   round(balanced_accuracy_score(y_test, y_pred), 4),
        })
        entrenados[nombre] = {'pipe': pipe, 'y_prob': y_prob, 'y_pred': y_pred}
        print(f'    AUC-PR={res_test[-1]["AUC_PR"]:.4f} Recall={res_test[-1]["Recall"]:.4f}')
        limpiar_memoria()
        memoria_usada()
    tabla_test = pd.DataFrame(res_test).sort_values('AUC_PR', ascending=False)
    mejor      = tabla_test.iloc[0]['Modelo']
    guardar_checkpoint({'tabla_test': tabla_test,
        'entrenados': entrenados, 'mejor': mejor},
        CP_EVAL, 'evaluacion final DSSA')
    display(tabla_test)
    tabla_test.to_csv(OUT_TAB / 'test_DSSA.csv', index=False)

colores_top = [COLORS['sin_signos'], COLORS['primary'], COLORS['accent']]
fig, axes   = plt.subplots(1, 3, figsize=(18, 5))
for i, (nombre, res) in enumerate(entrenados.items()):
    RocCurveDisplay.from_predictions(
        y_test, res['y_prob'], name=nombre, color=colores_top[i], ax=axes[0])
    PrecisionRecallDisplay.from_predictions(
        y_test, res['y_prob'], name=nombre, color=colores_top[i], ax=axes[1])
axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_title('A. Curvas ROC'); axes[0].grid(alpha=0.3)
axes[1].axhline(y_test.mean(), linestyle='--', color='gray', alpha=0.5,
    label=f'Prevalencia: {y_test.mean():.1%}')
axes[1].set_title('B. Precision-Recall (primaria)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
ConfusionMatrixDisplay.from_predictions(
    y_test, entrenados[mejor]['y_pred'],
    display_labels=['No hosp.', 'Hospitalizado'],
    cmap='Blues', ax=axes[2], colorbar=False)
axes[2].set_title(
    f'C. Confusion — {mejor}\nFN=caso grave no detectado')
plt.suptitle(f'Evaluacion final · DSSA | Mejor: {mejor}',
    fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIG / 'sin_signos_evaluacion_test.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
memoria_usada()

## 6. Importancia de variables

In [ ]:
CP_IMP = CHECKPOINTS / 'importancia_DSSA.pkl'
if checkpoint_existe(CP_IMP):
    print('Importancia ya calculada — cargando...')
    df_imp = joblib.load(CP_IMP)
else:
    pipe_m = entrenados[mejor]['pipe']
    print(f'Calculando importancia: {mejor}...')
    perm = permutation_importance(pipe_m, X_test, y_test,
        n_repeats=20, random_state=RANDOM_STATE, scoring='average_precision', n_jobs=-1)
    df_imp = pd.DataFrame({'Variable': FEATURES,
        'Importancia': perm.importances_mean, 'Std': perm.importances_std
    }).sort_values('Importancia', ascending=False)
    df_imp['Tipo'] = ['Signo de alarma' if v in SIGNOS_ALARMA
        else ('Sintoma general' if v in SINTOMAS_GENERALES else 'Demo/Temporal')
        for v in df_imp['Variable']]
    guardar_checkpoint(df_imp, CP_IMP, 'importancia')
    limpiar_memoria(perm)

color_tipo = {'Signo de alarma': COLORS['con_signos'],
              'Sintoma general': COLORS['sin_signos'], 'Demo/Temporal': COLORS['accent']}
df_top = df_imp.head(15)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(df_top['Variable'][::-1], df_top['Importancia'][::-1],
    xerr=df_top['Std'][::-1],
    color=[color_tipo[t] for t in df_top['Tipo']][::-1],
    capsize=4, alpha=0.87, edgecolor='white')
ax.legend(handles=[mpatches.Patch(color=c,label=t) for t,c in color_tipo.items()], loc='lower right')
ax.set_xlabel('Reduccion en AUC-PR (permutation importance)')
ax.set_title(f'Importancia — {mejor} · DSSA')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIG/'sin_signos_importancia.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
df_imp.to_csv(OUT_TAB/'importancia_DSSA.csv', index=False)
guardar_checkpoint({'modelo': mejor, 'pipeline': entrenados[mejor]['pipe'],
    'features': FEATURES, 'tabla_test': tabla_test, 'importancia': df_imp},
    CHECKPOINTS/'mejor_modelo_DSSA.pkl', 'modelo final')
print(f'Listo: {mejor}')
memoria_usada()

---

## Resumen DSSA

| Aspecto | Detalle |
|---------|--------|
| Cohorte | 15,734 casos · particion 80/20 estratificada |
| Metrica primaria | AUC-PR (desbalance severo 96/4) |
| SMOTE k_neighbors | 3 (mas conservador por pocos positivos) |
| Evaluacion final | Una unica evaluacion sobre test set |